In [1]:
!pip uninstall -y transformers peft accelerate
!pip install transformers==4.38.2
!pip install accelerate==0.27.2
!pip install peft==0.8.2
!pip install sentencepiece datasets evaluate -q

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Success

In [2]:
# =========================================================
# IMPORT LIBRARY
# =========================================================

import pandas as pd
import shutil

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    MT5Tokenizer,
    MT5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [3]:
# =========================================================
# HAPUS CHECKPOINT LAMA
# =========================================================

shutil.rmtree("./mt5-qg", ignore_errors=True)

In [5]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("H5_dataset_ML.csv")

In [6]:
# =========================================================
# BERSIHKAN DATASET
# =========================================================

df["input"] = df["input"].astype(str).str.strip()
df["target"] = df["target"].astype(str).str.strip()

# hapus data kosong
df = df[
    (df["input"] != "") &
    (df["target"] != "")
]

# hapus duplikat
df = df.drop_duplicates(
    subset=["input", "target"]
)

df = df.reset_index(drop=True)

In [7]:
# =========================================================
# INFO DATASET
# =========================================================

print("=" * 50)
print("JUMLAH DATASET")
print("=" * 50)

print(len(df))

print("\nCONTOH DATA:")
print(df.head())

JUMLAH DATASET
4674

CONTOH DATA:
                                               input  \
0   generate siapa: Pagi itu Rina bangun lebih awal.   
1   generate kapan: Pagi itu Rina bangun lebih awal.   
2  generate siapa: Rina merapikan tempat tidur di...   
3  generate apa: Rina merapikan tempat tidur di k...   
4  generate dimana: Rina merapikan tempat tidur d...   

                                            target  
0           Siapa yang bangun pagi itu lebih awal?  
1                              Kapan Rina bangun ?  
2  Siapa yang merapikan tempat tidur di kamar nya?  
3              Apa yang Rina rapikan di kamar nya?  
4             Di mana Rina merapikan tempat tidur?  


In [8]:
# =========================================================
# SPLIT DATASET
# 80% TRAIN
# 10% VALID
# 10% TEST
# =========================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("\nTRAIN :", len(train_df))
print("VALID :", len(valid_df))
print("TEST  :", len(test_df))


TRAIN : 3739
VALID : 467
TEST  : 468


In [9]:
# =========================================================
# CONVERT KE HUGGINGFACE DATASET
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [10]:
# =========================================================
# LOAD MODEL mT5-small
# =========================================================

model_name = "google/mt5-small"

tokenizer = MT5Tokenizer.from_pretrained(model_name)

model = MT5ForConditionalGeneration.from_pretrained(
    model_name
)

# penting
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
# =========================================================
# TOKENISASI
# =========================================================

max_input_length = 128
max_target_length = 64


def preprocess_function(examples):

    # tambahkan prefix task
    inputs = [
        "generate question: " + str(text)
        for text in examples["input"]
    ]

    # tokenize input
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    # tokenize target
    labels = tokenizer(
        [str(t) for t in examples["target"]],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # ubah padding menjadi -100
    # agar tidak dihitung saat loss
    label_ids = labels["input_ids"]

    label_ids = [
        [
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ]
        for label in label_ids
    ]

    model_inputs["labels"] = label_ids

    return model_inputs

In [12]:
# =========================================================
# TOKENIZE DATASET
# =========================================================

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/3739 [00:00<?, ? examples/s]

Map:   0%|          | 0/467 [00:00<?, ? examples/s]

Map:   0%|          | 0/468 [00:00<?, ? examples/s]

In [13]:
# =========================================================
# CEK HASIL TOKENISASI
# =========================================================

print("\nHASIL TOKENISASI:")
print(tokenized_train[0])

print("\nLABEL SAMPLE:")
print(tokenized_train[0]["labels"][:20])


HASIL TOKENISASI:
{'input': 'generate apa: Vina menemukan empat sudut gambar.', 'target': 'Apa yang Vina temukan?', '__index_level_0__': 2715, 'input_ids': [259, 66792, 7680, 267, 259, 66792, 3271, 267, 67827, 10125, 16310, 35840, 259, 263, 153875, 17418, 260, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

In [14]:
# =========================================================
# TRAINING ARGUMENT
# =========================================================

training_args = Seq2SeqTrainingArguments(

    output_dir="./mt5-qg",

    evaluation_strategy="epoch",

    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=3,

    predict_with_generate=True,

    logging_steps=10,

    fp16=False
)

In [15]:
# =========================================================
# TRAINER
# =========================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    tokenizer=tokenizer
)


In [16]:
# =========================================================
# TRAIN MODEL
# =========================================================

trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,2.478600,1.278531
2,1.660700,0.928091
3,1.399100,0.823571


TrainOutput(global_step=2805, training_loss=3.8761222451765907, metrics={'train_runtime': 848.4202, 'train_samples_per_second': 13.221, 'train_steps_per_second': 3.306, 'total_flos': 1482746320650240.0, 'train_loss': 3.8761222451765907, 'epoch': 3.0})

In [17]:
# =========================================================
# SIMPAN MODEL
# =========================================================

trainer.save_model("model_qg_mt5")

tokenizer.save_pretrained("model_qg_mt5")

print("\nMODEL BERHASIL DISIMPAN")


MODEL BERHASIL DISIMPAN


In [24]:
import torch

text = "Rina membaca buku di perpustakaan."

input_text = "generate Apa: " + text

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

device = model.device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

output_ids = model.generate(

    **inputs,

    max_length=32,

    num_beams=5,

    early_stopping=True,

    no_repeat_ngram_size=2,

    repetition_penalty=2.5,

    length_penalty=1.0
)

hasil = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("=" * 50)
print("HASIL GENERATE")
print("=" * 50)

print("INPUT :")
print(text)

print("\nOUTPUT :")
print(hasil)

HASIL GENERATE
INPUT :
Rina membaca buku di perpustakaan.

OUTPUT :
<pad> Apa yang Rina baca di perpustakaan? Rin membaca buku di Perpustakaan.?Rina mebaca buku? Riapa yang mengambil buku


In [25]:
import torch

text = "Rina membaca buku di perpustakaan."

input_text = (
    "generate siapa: "
    + text
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

device = model.device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

output_ids = model.generate(

    **inputs,

    max_new_tokens=20,

    num_beams=5,

    early_stopping=True,

    no_repeat_ngram_size=3,

    repetition_penalty=3.0,

    length_penalty=1.2,

    eos_token_id=tokenizer.eos_token_id,

    pad_token_id=tokenizer.pad_token_id
)

hasil = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

print(hasil)

<pad> Apa yang Rina baca? Rina membaca buku di perpustakaan? rina membaca
